# Industrial Cable Defect Detection using Autoencoders
######

## Project Report
#### 50.039 Theory and Practice of Deep Learning (2026)
###### Deadline: 18 April 2026, 11:59 PM

####

#### Group 7
- Ang Li En Eldrick (1006908)
- Malvin Ken Sudirgo (1007164)
- Toh Jia Le (1007004)

######
**GitHub Repository:** https://github.com/jialetoh/50039-proj-group07-2026

---

## 1. Project Topic

### 1.1 Problem
In industrial manufacturing, manual inspection of cables for structural integrity is prone to human error and fatigue. Inspection mistakes can lead to system instability and sensor failure, potentially causing injuries and even death.

### 1.2 Objective
This project aims to do **pixel-level anomaly detection** and **binary segmentation** of cable defects, to potentially automate cable inspections using deep learning. Using an **unsupervised approach**, models will be trained on normal cable images only. Then, they will be evaluated based on how well they can detect and identify anomalous regions in defective cable images.

---

## 2. Dataset

### 2.1 Dataset Overview

This project uses the **cable** category from the **MVTec Anomaly Detection (MVTec AD)** dataset.

- Dataset obtained from Kaggle (https://www.kaggle.com/datasets/ipythonx/mvtec-ad?select=cable)
- More information available at the official MVTec website (https://www.mvtec.com/company/research/datasets/mvtec-ad).
- More detailed information about the dataset can be found in `notebooks/01_data_exploration.ipynb`.

### 2.2 Dataset Composition

The cable category contains a total of **374 samples**:

- 282 are **normal** samples of defect-free cables.
- 92 are **anomalous** samples of cables containing visible defects.

All samples consist of one **3-channel** (RGB) PNG image of a cable, with a resolution of 1024×1024 pixels. Additionally, every anomalous sample has a corresponding **ground-truth segmentation mask** showing exactly where the defects are in the cable image. Each mask is a 1-channel grayscale PNG image at the same resolution (1024×1024), stored in the `data/cable/ground_truth/` directory.

### 2.3 Defect Categories

The 92 anomalous samples are divided into **8 different types of defects**, with 10 to 14 samples per defect category:

1. bent_wire
2. cable_swap
3. cut_inner_insulation
4. cut_outer_insulation
5. missing_cable
6. missing_wire
7. poke_insulation
8. combined

Figure 1 below provides a visual overview of the dataset. Three example images are shown for each defect category (and normal cables are labelled as "good").

<div style="text-align: center;">
    <img src="../figures/Fig1_dataset_overview.png" style="display: block; margin: 0 auto; max-width: 80%;">
    <br>
    <i>Figure 1. Overview of normal and anomalous cable samples across all eight defect categories.</i>
    <br>
</div>

Figure 2 below shows anomalous samples from each category and how their corresponding ground-truth segmentation mask highlights defective regions in the image.

<div style="text-align: center;">
    <img src="../figures/Fig2_segmask_preview.png" style="display: block; margin: 0 auto; max-width: 80%;">
    <br>
    <i>Figure 2. Anomalous cable samples from each defect category with their corresponding ground-truth segmentation masks.</i>
    <br>
</div>

### 2.4 Original Dataset Split

# **====== continue editing here. above this section is already done =======**

The dataset provided is already split into two sets:

- Train: 224 normal samples
- Test: 58 normal + 92 anomalous samples

This follows a standard Only normal images are used for training in an unsupervised setup, so a model only learns what is "normal", allowing it to flag anomalous images.



### 2.5 Relevance to Project Theme

#### Class Imbalance
The dataset exhibits natural class imbalance: only ~25% of the full dataset consists of anomalous samples (92 out of 374 total). Within the training set, this imbalance is absolute — there are zero anomalous images, with the model trained purely on 224 normal samples.

This mirrors real industrial inspection settings, where defective products are rare relative to conforming ones. The extreme imbalance motivates the use of unsupervised anomaly detection rather than binary classification: since defect examples are scarce and expensive to collect, a model must learn the distribution of normality and flag deviations, rather than fitting a balanced labelled dataset.

#### Real-World Relevance
The MVTec AD dataset is explicitly designed to replicate industrial optical inspection tasks. Defects such as cuts, bent wires, missing components, and combined faults are physical, real-world manufacturing errors captured under controlled industrial conditions — not synthetic digital artifacts.

The dataset is small by deep learning standards (~374 total samples, 10–14 per defect type), reflecting the difficulty of collecting and labelling defect images in practice. This makes supervised defect classification infeasible, as each defect type has insufficient samples for reliable training. Our unsupervised approach — training only on normal samples — is therefore both practically motivated and well-suited to the dataset's constraints.

## 3. Approach

### 3.1 Dataset Split
- Training set: normal images only
- Validation set: 15% split from the normal training images
- Test set: official MVTec cable test set containing both normal and anomalous images

MVTec cable subset provides only training and test splits, so the original training set of normal images was further divided into training and validation subsets for model development.

- **Training set:** 224 normal images  
- **Validation set:** 58 normal images  
- **Test set:** 58 normal images and 92 anomalous images  

The validation set was used to monitor reconstruction performance during training, while the test set was used for final evaluation of anomaly detection and pixel-level defect segmentation.

### 3.2 Dataset Preprocessing

All images were resized from their native resolution of 1024×1024 to **256×256 pixels** to reduce memory and computational cost while retaining sufficient detail for reconstruction.

Pixel values were normalised to the range **[0, 1]** by dividing by 255.

**Baseline model:** trained without data augmentation. No augmentation was applied to validation or test sets in any experiment.

**Later experiments:** mild augmentation was explored on the training set to assess its effect on model generalisation. Augmentations included:
- Random rotation (±5°)
- Random affine translation (±5% per axis)
- ColorJitter (brightness, contrast, saturation, and hue each ±5%)

Augmentation did not improve anomaly detection performance and was excluded from the final best model. This could be due to the similarity in appearance across the normal samples, where mild augmentations may not have added meaningful variability to the training data.

**Background masking:** A per-image cable mask was generated using HSV saturation thresholding (saturation > 0.15), followed by morphological closing and dilation to fill gaps. This mask was explored as a way to restrict loss computation to cable pixels and ignore the grey background. However, this strategy did not consistently improve performance and was not applied in the final best model.

### 3.3 Inputs and Outputs
**Input:**  
- 1024x1024 RGB image (3-channel)

**Output:**  
- 1024x1024 binary segmentation mask, where a pixel value of 0 denotes a normal
background region and 1 denotes an anomalous/defective region.

### 3.3 Model Architecture

#### 3.3.1 Inspiration for Our Approach

Since the training set contains only normal images, supervised defect classification is not possible. The unsupervised anomaly detection paradigm is well-suited to this constraint: a model is trained to reconstruct normal inputs accurately. At test time, anomalous regions are expected to be reconstructed poorly, producing higher reconstruction error that can localise defects.

Convolutional autoencoders are a natural baseline for this task — they learn compact spatial representations via an encoder-decoder structure. We started with a simple custom convolutional autoencoder to establish a baseline. Motivated by the limited training data (224 images) and the representational power of features from large-scale datasets, we then introduced a pretrained ResNet18 encoder as a stronger feature extractor, expected to provide richer representations without requiring large amounts of task-specific data.

#### 3.3.2 Baseline Model

The baseline is a symmetric 3-layer convolutional autoencoder. The encoder reduces spatial resolution while increasing channel depth; the decoder mirrors this, upsampling back to the original resolution. Model definition is in `src/models.py`.

| Stage | Layer | Output Shape |
|-------|-------|-------------|
| Input | — | [B, 3, 256, 256] |
| Encoder Conv1 | Conv2d(3→16, 3×3) + BN + ReLU + MaxPool(2,2) | [B, 16, 128, 128] |
| Encoder Conv2 | Conv2d(16→32, 3×3) + BN + ReLU + MaxPool(2,2) | [B, 32, 64, 64] |
| Encoder Conv3 | Conv2d(32→64, 3×3) + BN + ReLU + MaxPool(2,2) | [B, 64, 32, 32] |
| Bottleneck | — | [B, 64, 32, 32] |
| Decoder DeConv3 | ConvTranspose2d(64→32, 3×3, stride=2) + BN + ReLU | [B, 32, 64, 64] |
| Decoder DeConv2 | ConvTranspose2d(32→16, 3×3, stride=2) + BN + ReLU | [B, 16, 128, 128] |
| Decoder DeConv1 | ConvTranspose2d(16→3, 3×3, stride=2) + Sigmoid | [B, 3, 256, 256] |

#### 3.3.2.1 Baseline Model (4-layers)

The baseline model was also implemented with 4 convolutional layers in the encoder and decoder, with a bottleneck size of 128 channels. However, this deeper architecture did not improve performance compared to the 3-layer version, likely due to overfitting given the limited training data. The 3-layer architecture was therefore chosen as the baseline for subsequent experiments.

| Stage | Layer | Output Shape |
|-------|-------|-------------|
| Input | — | [B, 3, 256, 256] |
| Encoder Conv1 | Conv2d(3→16, 3×3) + BN + ReLU + MaxPool(2,2) | [B, 16, 128, 128] |
| Encoder Conv2 | Conv2d(16→32, 3×3) + BN + ReLU + MaxPool(2,2) | [B, 32, 64, 64] |
| Encoder Conv3 | Conv2d(32→64, 3×3) + BN + ReLU + MaxPool(2,2) | [B, 64, 32, 32] |
| Encoder Conv4 | Conv2d(64→128, 3×3) + BN + ReLU + MaxPool(2,2) | [B, 128, 16, 16] |
| Bottleneck | — | [B, 128, 16, 16] |
| Decoder DeConv4 | ConvTranspose2d(128→64, 3×3, stride=2) + BN + ReLU | [B, 64, 32, 32] |
| Decoder DeConv3 | ConvTranspose2d(64→32, 3×3, stride=2) + BN + ReLU | [B, 32, 64, 64  ] |
| Decoder DeConv2 | ConvTranspose2d(32→16, 3×3, stride=2) + BN + ReLU | [B, 16, 128, 128] |
| Decoder DeConv1 | ConvTranspose2d(16→3, 3×3, stride=2) + Sigmoid | [B, 3, 256, 256] |

#### 3.3.3 Improved Model

The improved model replaces the custom encoder with a **pretrained ResNet18** backbone (ImageNet weights). The first seven layers of ResNet18 produce feature maps of shape [B, 256, 16, 16]. A bottleneck convolutional block follows, and a four-stage transposed convolutional decoder upsamples back to 256×256. Model definition is in `src/models_pretrained.py`.

| Stage | Layer | Output Shape |
|-------|-------|-------------|
| Input | — | [B, 3, 256, 256] |
| Encoder | ResNet18 (first 7 layers, pretrained ImageNet) | [B, 256, 16, 16] |
| Bottleneck | Conv2d(256→256, 3×3) + BN + LeakyReLU | [B, 256, 16, 16] |
| Decoder Stage 1 | ConvTranspose2d(256→128, 3×3, stride=2) + BN + LeakyReLU | [B, 128, 32, 32] |
| Decoder Stage 2 | ConvTranspose2d(128→64, 3×3, stride=2) + BN + LeakyReLU | [B, 64, 64, 64] |
| Decoder Stage 3 | ConvTranspose2d(64→32, 3×3, stride=2) + BN + LeakyReLU | [B, 32, 128, 128] |
| Decoder Stage 4 | ConvTranspose2d(32→16, 3×3, stride=2) + BN + LeakyReLU | [B, 16, 256, 256] |
| Output Head | Conv2d(16→3, 3×3) + Sigmoid | [B, 3, 256, 256] |

Two variants were evaluated:
- **Frozen encoder:** ResNet18 weights fixed; only bottleneck and decoder trained (~95K trainable parameters).
- **Fine-tuned encoder:** All weights trainable (~11.2M parameters).

### 3.5 Model Training

All models were trained using the **Adam optimizer** on normal images only. Training was monitored via validation reconstruction loss, with early stopping (patience = 15 epochs) to prevent overfitting, though there were no encounters of early stopping due to the small size of our dataset, which limited the model's ability to overfit. The best checkpoint per run is saved to `checkpoints/`.

| Setting | Baseline (Notebook 02) | Pretrained (Notebook 03) | Best Tuned (Notebook 04) |
|---------|----------------------|--------------------------|--------------------------|
| Epochs | 100 | 50 | 50 |
| Learning rate | 1e-3 | 1e-4 | 1e-3 |
| Batch size | 16 | 16 | 16 |
| Early stopping patience | 15 | 15 | 15 |
| Encoder frozen | N/A | Both variants | Yes (frozen) |

Training code is in `02_baseline_autoencoder.ipynb`, `03_pretrained_encoder.ipynb`, and `04_tuned_pretrained_encoder.ipynb`.

Initial training of the baseline model revealed significantly low performance (ROC-AUC = 0.5555, PR-AUC = 0.6822), motivating the introduction of a pretrained encoder and subsequent hyperparameter tuning to improve anomaly detection capabilities.

#### Baseline Convolutional AutoEncoder Performance
| Metric | Baseline |
|---|---:|
| ROC-AUC | 0.5555 |
| PR-AUC | 0.6822 |
| F1 | 0.3577 |
| Recall | 0.2391 |
| Precision | 0.7097 |
| Accuracy | 0.4733 |
| Balanced Acc | 0.5420 |
| FPR | 0.1552 |
| FNR | 0.7609 |
| Threshold | 0.1221 |
| Pixel AUROC | 0.4768 |
| Mean IoU | 0.0493 |


We then proceed to a pretrained encoder (transfer learning), which significantly improved performance (ROC-AUC = 0.7150, PR-AUC = 0.7508) even without tuning. However, there is still room for improvement, which motivates the hyperparameter tuning experiments in the next section (3.6).

| Metric | Frozen | Fine-tune |
|---|---:|---:|
| ROC-AUC | 0.7150 | 0.6143 |
| PR-AUC | 0.8238 | 0.7580 |
| F1 | 0.6309 | 0.5286 |
| Recall | 0.5109 | 0.4022 |
| Precision | 0.8246 | 0.7708 |
| Balanced Acc | 0.6692 | 0.6063 |
| FPR | 0.1724 | 0.1897 |
| FNR | 0.4891 | 0.5978 |
| Pixel AUROC | 0.5328 | 0.5579 |
| Mean IoU | 0.0522 | 0.0674 |

#### Baseline Pretrained ResNet18 Encoder Performance

### 3.6 Hyperparameter Search / Tuning (Done)

### Experimental Configurations  

| Hyperparameter            | Values Tested        |
|---------------------|---------------------|
| Learning Rate       | 1e-4, 1e-3          |
| MAE Weight ($\lambda$) | 0.4, 0.6            |
| Bottleneck Size     | 128, 256            |
| Encoder Frozen      | True, False         |
| Epochs              | 50 (fixed)          |
| Patience            | 15 (fixed)          |

Configurations were ranked using a weighted composite test score:

**Score = 0.30 × ROC-AUC + 0.30 × PR-AUC + 0.15 × Recall + 0.15 × Precision + 0.10 × F1**

Our main focus was on the ROC-AUC and PR-AUC metrics, as they are threshold-independent and provide a robust measure of the model’s ability to distinguish between normal and anomalous samples across all possible thresholds. Recall and Precision were also important for understanding the model’s performance in terms of correctly identifying defects (Recall) while minimizing false positives (Precision). The F1 score provided a balanced measure of both aspects. Hyperparameters were tuned in `04_tuned_pretrained_encoder.ipynb`.

**Summary:**  
A total of **16 configurations** were evaluated using grid search over all parameter combinations.

The best performing model from the experiment above was later tested without any augmentations.

After this, experiments were done to explore the effect of **MAE weighting and learning rate** with a fixed bottleneck size (256) and a frozen encoder as shown in the table below. Training epochs were increased to 100 with patience 15.

### Subsequent Experimental Configurations  

| Config | Learning Rate | MAE Weight | Bottleneck Size | Encoder Frozen | Epochs | Patience |
|-------|--------------|-----------|----------------|----------------|--------|----------|
| 1 | 1e-4 | 1.0 | 256 | True  | 100 | 15 |
| 2 | 1e-4 | 0.4 | 256 | True  | 100 | 15 |
| 3 | 1e-3 | 1.0 | 256 | True  | 100 | 15 |
| 4 | 1e-3 | 0.4 | 256 | True  | 100 | 15 |

### 3.7 Loss Functions (Done)
The model is trained using a composite reconstruction loss that combines pixel-wise accuracy and structural similarity:

$$
\mathcal{L} = \lambda \cdot \mathcal{L}_{\text{MAE}} + (1 - \lambda)\cdot (1 - \text{SSIM})
$$

where λ (mae_weight) controls the trade-off between the two components.

MAE (L1 Loss): Measures pixel-wise differences between the reconstructed output and ground truth.
SSIM Loss: Defined as 1−SSIM, encouraging structural and perceptual similarity.

When masking is enabled, both MAE and SSIM are computed only over a region of interest using a binary mask, focusing the model on relevant areas. Otherwise, the loss is computed over the full image.

No additional auxiliary losses are used.

### 3.8 Evaluation Metrics (Done)
Explain the metrics used to evaluate performance.
State clearly how thresholds were chosen for binary segmentation.

To comprehensively evaluate the performance of the binary segmentation model, multiple metrics were employed to capture different aspects of prediction quality, particularly under potential class imbalance.

**Threshold-independent metrics were first considered to assess the model’s ranking ability:**

ROC-AUC (Area Under the Receiver Operating Characteristic Curve) measures the model’s ability to distinguish between positive and negative classes across all possible thresholds. A higher ROC-AUC indicates better separability.
PR-AUC (Average Precision) summarizes the Precision–Recall curve and is especially informative for imbalanced datasets, where correctly identifying the positive class is more critical.

**Threshold-dependent metrics were then used to evaluate classification performance after converting predicted probabilities into binary outputs:**

F1-score (F1) is the harmonic mean of precision and recall, providing a balanced measure when both false positives and false negatives are important.
Recall (Sensitivity) measures the proportion of actual positives correctly identified.
Precision measures the proportion of predicted positives that are truly positive.
Accuracy represents the overall proportion of correct predictions.
Balanced Accuracy (BAcc) computes the average of recall obtained on each class, making it robust to class imbalance.

**Error-specific metrics were also included to better understand failure modes:**

False Positive Rate (FPR) quantifies the proportion of negative samples incorrectly classified as positive.
False Negative Rate (FNR) measures the proportion of positive samples incorrectly classified as negative.

**Threshold Selection Strategy**

Since binary segmentation requires converting predicted probabilities into binary masks, a decision threshold was applied. The reference threshold was selected based on validation performance to ensure a principled trade-off between true positive and false positive rates.

Specifically, the threshold was chosen as the value that maximized the ROC-AUC on the validation set. This approach was adopted because:

The task benefits from strong overall class separability across different threshold settings.
ROC-AUC evaluates the model’s ability to distinguish between classes independent of any single operating point.
It accounts for the trade-off between True Positive Rate (TPR) and False Positive Rate (FPR), making it suitable when both types of errors are important.

---

## Results
▪ Visualization of model performance
• Accuracy and loss curves
• Performance on validation set images


### 4.1 Training Curves
Include:
- training loss curve
- validation loss curve

Comment on:
- convergence
- overfitting / underfitting
- stability of training

### 4.2 Qualitative Results
Show example outputs on validation / test images.

Include:
- original image
- ground truth mask
- reconstruction
- anomaly map
- predicted binary mask

### 4.3 Quantitative Results (Done for tuning)
Present final metrics in a table.

**16 Configurations experiment:**
<div style="overflow-x:auto;">
<table border="1" cellpadding="4" cellspacing="0">
<thead>
<tr>
  <th>Metric</th>
  <th>lr=0.0001, MAE weightage=0.4, Bottleneck Width=128, Weight Freeze=False</th>
  <th>lr=0.0001, MAE weightage=0.4, Bottleneck Width=128, Weight Freeze=True</th>
  <th>lr=0.0001, MAE weightage=0.4, Bottleneck Width=256, Weight Freeze=False</th>
  <th>lr=0.0001, MAE weightage=0.4, Bottleneck Width=256, Weight Freeze=True</th>
  <th>lr=0.0001, MAE weightage=0.6, Bottleneck Width=128, Weight Freeze=False</th>
  <th>lr=0.0001, MAE weightage=0.6, Bottleneck Width=128, Weight Freeze=True</th>
  <th>lr=0.0001, MAE weightage=0.6, Bottleneck Width=256, Weight Freeze=False</th>
  <th>lr=0.0001, MAE weightage=0.6, Bottleneck Width=256, Weight Freeze=True</th>
  <th>lr=0.001, MAE weightage=0.4, Bottleneck Width=128, Weight Freeze=False</th>
  <th>lr=0.001, MAE weightage=0.4, Bottleneck Width=128, Weight Freeze=True</th>
  <th>lr=0.001, MAE weightage=0.4, Bottleneck Width=256, Weight Freeze=False</th>
  <th>lr=0.001, MAE weightage=0.4, Bottleneck Width=256, Weight Freeze=True</th>
  <th>lr=0.001, MAE weightage=0.6, Bottleneck Width=128, Weight Freeze=False</th>
  <th>lr=0.001, MAE weightage=0.6, Bottleneck Width=128, Weight Freeze=True</th>
  <th>lr=0.001, MAE weightage=0.6, Bottleneck Width=256, Weight Freeze=False</th>
  <th>lr=0.001, MAE weightage=0.6, Bottleneck Width=256, Weight Freeze=True</th>
</tr>
</thead>
<tbody>
<tr>
  <td>ROC-AUC</td>
  <td>0.6361</td><td>0.6434</td><td>0.6402</td><td>0.6239</td>
  <td>0.6282</td><td>0.6278</td><td>0.6569</td><td>0.7442</td>
  <td>0.5508</td><td>0.7204</td><td>0.5440</td><td>0.7661</td>
  <td>0.5538</td><td>0.7180</td><td>0.5500</td><td>0.7397</td>
</tr>
<tr>
  <td>PR-AUC</td>
  <td>0.7470</td><td>0.7466</td><td>0.7434</td><td>0.7355</td>
  <td>0.7438</td><td>0.7471</td><td>0.7599</td><td>0.8433</td>
  <td>0.7096</td><td>0.8170</td><td>0.6963</td><td>0.8409</td>
  <td>0.7145</td><td>0.8160</td><td>0.7043</td><td>0.8342</td>
</tr>
<tr>
  <td>F1</td>
  <td>0.3937</td><td>0.4462</td><td>0.3833</td><td>0.4186</td>
  <td>0.4394</td><td>0.4627</td><td>0.5850</td><td>0.5890</td>
  <td>0.4194</td><td>0.6174</td><td>0.3636</td><td>0.6528</td>
  <td>0.4355</td><td>0.5441</td><td>0.4160</td><td>0.6099</td>
</tr>
<tr>
  <td>Recall</td>
  <td>0.2717</td><td>0.3152</td><td>0.2500</td><td>0.2935</td>
  <td>0.3152</td><td>0.3370</td><td>0.4674</td><td>0.4674</td>
  <td>0.2826</td><td>0.5000</td><td>0.2391</td><td>0.5109</td>
  <td>0.2935</td><td>0.4022</td><td>0.2826</td><td>0.4674</td>
</tr>
<tr>
  <td>Precision</td>
  <td>0.7143</td><td>0.7632</td><td>0.8214</td><td>0.7297</td>
  <td>0.7250</td><td>0.7381</td><td>0.7818</td><td>0.7963</td>
  <td>0.8125</td><td>0.8070</td><td>0.7586</td><td>0.9038</td>
  <td>0.8438</td><td>0.8409</td><td>0.7879</td><td>0.8776</td>
</tr>
<tr>
  <td>Accuracy</td>
  <td>0.4867</td><td>0.5200</td><td>0.5067</td><td>0.5000</td>
  <td>0.5067</td><td>0.5200</td><td>0.5933</td><td>0.6000</td>
  <td>0.5200</td><td>0.6200</td><td>0.4867</td><td>0.6667</td>
  <td>0.5333</td><td>0.5867</td><td>0.5133</td><td>0.6333</td>
</tr>
<tr>
  <td>Balanced Acc</td>
  <td>0.5497</td><td>0.5800</td><td>0.5819</td><td>0.5605</td>
  <td>0.5628</td><td>0.5737</td><td>0.6302</td><td>0.6389</td>
  <td>0.5896</td><td>0.6552</td><td>0.5592</td><td>0.7123</td>
  <td>0.6036</td><td>0.6407</td><td>0.5810</td><td>0.6820</td>
</tr>
<tr>
  <td>Threshold</td>
  <td>0.4357</td><td>0.4412</td><td>0.4556</td><td>0.4349</td>
  <td>0.4238</td><td>0.4313</td><td>0.3668</td><td>0.3571</td>
  <td>0.1538</td><td>0.2899</td><td>0.1423</td><td>0.2369</td>
  <td>0.1595</td><td>0.2668</td><td>0.1410</td><td>0.2432</td>
</tr>
<tr>
  <td>FPR</td>
  <td>0.1724</td><td>0.1552</td><td>0.0862</td><td>0.1724</td>
  <td>0.1897</td><td>0.1897</td><td>0.2069</td><td>0.1897</td>
  <td>0.1034</td><td>0.1897</td><td>0.1207</td><td>0.0862</td>
  <td>0.0862</td><td>0.1207</td><td>0.1207</td><td>0.1034</td>
</tr>
<tr>
  <td>FNR</td>
  <td>0.7283</td><td>0.6848</td><td>0.7500</td><td>0.7065</td>
  <td>0.6848</td><td>0.6630</td><td>0.5326</td><td>0.5326</td>
  <td>0.7174</td><td>0.5000</td><td>0.7609</td><td>0.4891</td>
  <td>0.7065</td><td>0.5978</td><td>0.7174</td><td>0.5326</td>
</tr>
</tbody>
</table>
</div>

**With/Without aug for the best configuration determined from above (lr0.001_mae0.4_b256_freezeTrue)**
Note that this had to be trained again, hence the difference in results

| Metric        | best_no_aug | best_with_aug |
|---------------|------------|---------------|
| ROC-AUC       | 0.6441     | 0.6357        |
| PR-AUC        | 0.7643     | 0.7577        |
| F1            | 0.5180     | 0.5286        |
| Recall        | 0.3913     | 0.4022        |
| Precision     | 0.7660     | 0.7708        |
| Accuracy      | 0.5533     | 0.5600        |
| Balanced Acc  | 0.6008     | 0.6063        |
| Threshold     | 0.4189     | 0.4199        |
| FPR           | 0.1897     | 0.1897        |
| FNR           | 0.6087     | 0.5978        |

**Subsequent experiments with 100 epochs (Best model determined as lr=0.001, MAE weightage=1, Bottleneck Width=256, Weight Freeze=True)**

| Metric        | lr=0.0001, MAE=1, Bottleneck Width=256, Freeze=False | lr=0.0001, MAE=0.4, Bottleneck Width=256, Freeze=False | lr=0.001, MAE=1, Bottleneck Width=256, Freeze=False | lr=0.001, MAE=0.4, Bottleneck Width=256, Freeze=False |
|---------------|-----------------------------------------------------|--------------------------------------------------------|---------------------------------------------------|-----------------------------------------------------|
| ROC-AUC       | 0.7195                                              | 0.7146                                                 | 0.7749                                            | 0.7805                                              |
| PR-AUC        | 0.7996                                              | 0.8181                                                 | 0.8518                                            | 0.8578                                              |
| F1            | 0.6946              | 0.6667                 | 0.7394             | 0.6800               |
| Recall        | 0.6304              | 0.5435                 | 0.6630             | 0.5543               |
| Precision     | 0.7733              | 0.8621                 | 0.8356             | 0.8793               |
| Accuracy      | 0.6600              | 0.6667                 | 0.7133             | 0.6800               |
| Balanced Acc  | 0.6687              | 0.7028                 | 0.7281             | 0.7168               |
| Threshold     | 0.0635              | 0.3017                 | 0.0507             | 0.2136               |
| FPR           | 0.2931              | 0.1379                 | 0.2069             | 0.1207               |
| FNR           | 0.3696              | 0.4565                 | 0.3370             | 0.4457               |


### 4.4 Performance on Validation Set Images
Briefly summarize how the model behaved on validation images and whether validation loss aligned with final anomaly detection performance.

---

## 5. Discussion

### 5.1 What Worked Well

**Transfer learning from ImageNet:** Replacing the custom encoder with a pretrained ResNet18 backbone produced the single largest performance gain, improving ROC-AUC from 0.5645 (baseline) to 0.7150 (frozen pretrained) — a gain of ~15 percentage points. ImageNet-pretrained features provide strong general visual representations that the custom encoder could not learn from 224 images.

**Frozen encoder as a regulariser:** Freezing the ResNet18 encoder and training only the bottleneck and decoder (~95K parameters) outperformed fine-tuning the full network (~11.2M parameters). With only 224 training images, the fine-tuned network overfits; the frozen encoder acts as a compact, stable feature extractor.

**MAE-only loss:** Removing SSIM from the loss (α = 1.0) consistently outperformed combined MAE+SSIM configurations. The simpler loss landscape facilitated faster convergence, particularly at higher learning rates.

### 5.2 Failure Cases / Malfunctioning Examples

**High false negative rate:** The best model misses ~42% of anomalous images (FNR = 0.4239). Defect types with subtle appearance differences from normal cables — such as `poke_insulation` or `cut_inner_insulation` — are particularly challenging, as reconstruction error may fall below the detection threshold.

**Baseline model failure:** The baseline ConvAutoencoder largely failed at anomaly detection (ROC-AUC = 0.5645, F1 = 0.0215). Its reconstruction capacity was insufficient to distinguish normal from anomalous images — in some cases, it reconstructed defective images just as well as normal ones, producing near-zero error on both and offering no detection signal. This is likely due to the limited number of training images and the model’s limited representational power, which prevented it from learning a robust representation of normality.

### 5.3 Comparison Against State-of-the-Art
Briefly compare your results with established methods.

Possible methods:
- PatchCore
- PaDiM
- FastFlow
- Anomalib baselines

Discuss:
- accuracy gap
- computational cost
- implementation complexity
- fairness of comparison

### 5.4 Limitations

- **Small training set:** With only 224 normal training images, learning a robust normality distribution is difficult, particularly for fine-grained texture and structural features.
- **Simple decoder design:** The transposed convolutional decoder lacks skip connections, limiting the precision of spatial reconstruction and the sharpness of anomaly maps.
- **Reconstruction-based scoring:** Averaging reconstruction error over all pixels gives equal weight to informative and uninformative regions. Small defects may not contribute enough to the image-level score to exceed the detection threshold.
- **Threshold sensitivity:** The FPR-constrained threshold was tuned on validation normal images only. Since the validation set contains no anomalous examples, the threshold cannot be directly optimised for recall or F1.
- **Background masking ineffective:** Despite the intuitive motivation of focusing the model on cable pixels, the masking strategy did not consistently improve performance and was excluded from the best model.

### 5.5 Future Work
- **Skip connections (U-Net decoder):** Encoder-to-decoder skip connections would allow the decoder to access fine-grained spatial features, potentially producing sharper anomaly maps with better pixel-level localisation.
- **Feature-level anomaly scoring:** Moving beyond pixel-space reconstruction to feature-space methods (e.g., PatchCore, PaDiM) would likely yield substantial gains. These methods achieve near-perfect AUROC on MVTec AD categories.
- **Ensemble methods:** Combining anomaly maps from multiple model variants may reduce false negatives through complementary error coverage.
---

## How to Run the Code

### Requirements
- Python 3 (tested on v3.13.4)
- OpenCV (tested on 4.13.0)
- PyTorch (tested on 2.10.0)
- Matplotlib (tested on v3.10.8)
- Numpy (tested on v2.4.1)

**Installing dependencies:** `pip install -r requirements.txt`


### Dataset Setup
1. Download the cable category from the MVTec AD dataset on Kaggle: https://www.kaggle.com/datasets/ipythonx/mvtec-ad?select=cable
2. Extract and place the dataset under `data/cable/` in the project root so that the structure is:

```
data/
  cable/
    train/
      good/          ← 224 normal training images
    test/
      good/          ← 58 normal test images
      bent_wire/
      cable_swap/
      cut_inner_insulation/
      cut_outer_insulation/
      missing_cable/
      missing_wire/
      poke_insulation/
      combined/
    ground_truth/
      bent_wire/
      cable_swap/
      ...            ← ground-truth binary masks for each defect category
```

### Running the Project

Run the notebooks in the following order
1. 01_data_exploration.ipynb
2. 02_baseline_autoencoder.ipynb
3. 03_pretrained_encoder.ipynb
4. 04_augmentation_and_tuning.ipynb
5. 05_final_report.ipynb

### Training from Scratch

Run all cells sequentially in notebooks `02`, `03`, and `04`. Each notebook:
1. Loads images from `data/cable/`
2. Initialises the model with the specified hyperparameters
3. Runs the training loop with early stopping (patience = 15)
4. Saves the best checkpoint (by validation loss) to `checkpoints/`

Training typically completes within 10–30 minutes per experiment on a GPU. Set `DEVICE = "cuda"` in each notebook to enable GPU acceleration.

### Loading a Saved Model

Pre-trained checkpoints are stored in `checkpoints/`:

| Checkpoint | Description |
|-----------|-------------|
| `baseline_autoencoder.pth` | Baseline 3-layer ConvAutoencoder |
| `resnet_frozen.pth` | ResNet18 with frozen encoder (Notebook 03) |
| `resnet_finetune.pth` | ResNet18 with fine-tuned encoder (Notebook 03) |
| `strong_mae-higher_lr-no_ssim.pth` | **Best model** — MAE-only, LR=1e-3, frozen encoder |

To reproduce evaluation results, run the evaluation cells in the relevant notebook after setting the `CHECKPOINT_PATH` variable to the desired checkpoint file.

## Contributions of each group member
- Ang Li En Eldrick
    - a
    - b
- Malvin Ken Sudirgo
    - a Report Writing
    - b Model Training and Fine Tuning
    - c Project Proposal
- Toh Jia Le
    - Project Proposal
    - b

## References

- Paul Bergmann, Michael Fauser, David Sattlegger, and Carsten Steger, "A Comprehensive Real-World Dataset for Unsupervised Anomaly Detection", IEEE Conference on Computer Vision and Pattern Recognition, 2019